# Chapter 4. Representing the Wireless Channel

The code from Chapter 4 of *Radio Propagation Ray Tracing with Sionna RT*.
The cells follow the order of the text, and the contents of each code cell are
the same as printed there.

Only the first Setup cell is not printed in the book. It performs the imports
and the scene loading that the printed code assumes. Once the setup cell has
run, the rest can be run from the top in order.

Fragments the text shows for explanation, such as a path to a scene of your own
or an object name that scene does not contain, will not run as they stand. A
note appears before each such cell.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sionna.rt as rt
from sionna.rt import (load_scene, scene as scene_assets,
                       Transmitter, Receiver, PlanarArray,
                       PathSolver, RadioMapSolver, Camera, RadioMaterial)

# the open street scene used in Chapters 3 and 4
scene = load_scene(scene_assets.simple_street_canyon)
scene.frequency = 3.5e9
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.add(Transmitter('tx0', position=[-70.0, -6.0, 10.0]))
scene.add(Receiver('rx0', position=[70.0, 6.0, 1.5]))

solver = PathSolver()
paths = solver(scene, max_depth=5)

## 4. The Channel Impulse Response (CIR)

In [ ]:
# extract the CIR from the paths as NumPy arrays
h, tau = paths.cir(normalize_delays=True, out_type='numpy')
print('h shape:', h.shape, 'tau shape:', tau.shape)

## 6. The Channel Frequency Response (CFR)

In [ ]:
import drjit as dr
import mitsuba as mi

# frequency offset relative to the carrier, [-50, 50] MHz
freq_offsets = dr.linspace(mi.Float, -50e6, 50e6, 1024)
H = paths.cfr(freq_offsets, normalize_delays=True,
              out_type='numpy')

## 8. Channel Taps

In [ ]:
# bandwidth 100 MHz, tap indices 0 to 63
taps = paths.taps(bandwidth=100e6, l_min=0, l_max=63,
                  out_type='numpy')
print('taps shape:', taps.shape)

## 10. Computing the Delay Spread

In [ ]:
def rms_delay_spread(a, tau):
    p = (abs(a)**2).flatten()
    t = tau.flatten()
    mean_tau = (p*t).sum() / p.sum()
    return np.sqrt((p*(t-mean_tau)**2).sum() / p.sum())

print(f'RMS delay spread: {rms_delay_spread(h, tau)*1e9:.2f} ns')

## 11. Reading the Angle Information

In [ ]:
# angles of arrival of the valid paths
valid = np.array(paths.valid).flatten()
phi_r = np.array(paths.phi_r).flatten()[valid]
theta_r = np.array(paths.theta_r).flatten()[valid]
print('theta [deg]:', np.degrees(theta_r))
print('phi [deg]:', np.degrees(phi_r))

## 12. Exporting to NumPy or PyTorch

In [ ]:
a_np, tau_np = paths.cir(out_type='numpy')
a_torch, tau_torch = paths.cir(out_type='torch')